# N-BaIoT — All-Device Preprocessing (Steps 1–4)

This notebook runs the complete preprocessing pipeline for every extracted device folder, **one device at a time**. It reads extracted CSVs only; it does not read or extract RAR archives.

**Steps:** 1) Load & inspect, 2) Clean, 3) Prepare labels, 4) Stratified 80:20 train/test split.

Design choices:
- Chunked CSV reading/writing to reduce RAM usage.
- A consistent global label mapping across all devices.
- Missing botnet folders are reported; available classes are still processed.
- Rows with invalid/missing feature values are dropped; labels are preserved.
- Duplicates are inspected within chunks but not removed.
- Splits are within-device, not cross-device evaluation.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

# CONFIGURATION
DATASET_ROOT = Path.cwd()   # Change this if the notebook is not in the main dataset folder
CHUNK_SIZE = 200_000
TRAIN_RATIO = 0.80
RANDOM_SEED = 42

LABELS = [
    "benign",
    "gafgyt_combo", "gafgyt_junk", "gafgyt_scan", "gafgyt_tcp", "gafgyt_udp",
    "mirai_ack", "mirai_scan", "mirai_syn", "mirai_udp", "mirai_udpplain"
]
LABEL_TO_ID = {name: i for i, name in enumerate(LABELS)}
ID_TO_LABEL = {i: name for name, i in LABEL_TO_ID.items()}

OUTPUT_ROOT = DATASET_ROOT / "preprocessed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Dataset root:", DATASET_ROOT.resolve())
print("Output root :", OUTPUT_ROOT.resolve())
print("Chunk size  :", f"{CHUNK_SIZE:,}")


## 1. Discover the device folders

The notebook searches immediate subfolders of `DATASET_ROOT` for `benign_traffic.csv`. Keep the notebook in the main dataset folder (the folder containing the nine device folders and `preprocessed`).


In [ ]:
def discover_devices(root):
    ignored = {"preprocessed", ".git", ".ipynb_checkpoints", "__pycache__"}
    found = []
    for folder in sorted(root.iterdir()):
        if folder.is_dir() and folder.name not in ignored:
            benign = list(folder.rglob("benign_traffic.csv"))
            if benign:
                found.append({"name": folder.name, "folder": folder, "benign": benign[0]})
    return found

devices = discover_devices(DATASET_ROOT)
print(f"Discovered {len(devices)} device folders:")
for d in devices:
    print(" ✓", d["name"], "->", d["benign"].relative_to(DATASET_ROOT))

if not devices:
    raise FileNotFoundError(
        "No device folders found. Set DATASET_ROOT to the directory containing the device folders."
    )
if len(devices) != 9:
    print(f"WARNING: expected 9 devices from the project plan, found {len(devices)}. Check DATASET_ROOT.")


## 2. Helper functions for extracted attack folders

Attack CSVs are discovered recursively inside folders named `gafgyt_attacks` and `mirai_attacks`. RAR files are ignored.


In [ ]:
def find_attack_csvs(device_folder, folder_name):
    paths = []
    for folder in device_folder.rglob(folder_name):
        if folder.is_dir():
            paths.extend(folder.glob("*.csv"))
    return sorted(set(p.resolve() for p in paths))

def get_sources(device_folder):
    benign_files = list(device_folder.rglob("benign_traffic.csv"))
    if not benign_files:
        raise FileNotFoundError(f"No benign_traffic.csv under {device_folder}")
    sources = [("benign", benign_files[0])]

    for botnet, folder_name in [("gafgyt", "gafgyt_attacks"), ("mirai", "mirai_attacks")]:
        files = find_attack_csvs(device_folder, folder_name)
        if not files:
            print(f"  WARNING: {folder_name} not found or contains no CSVs.")
        for path in files:
            label = f"{botnet}_{path.stem.lower()}"
            if label in LABEL_TO_ID:
                sources.append((label, path))
            else:
                print(f"  WARNING: skipping unexpected attack file {path.name}")
    return sources

def write_frame(frame, path, first):
    frame.to_csv(path, mode="w" if first else "a", header=first, index=False)


## 3. Process one device through Steps 1–4

Each device is completed before the next begins. Intermediate CSVs are written to disk in chunks rather than holding the entire device dataset in RAM.


In [ ]:
def process_device(info):
    name, folder = info["name"], info["folder"]
    out = OUTPUT_ROOT / name
    out.mkdir(parents=True, exist_ok=True)

    raw = out / "step1_raw_labelled.csv"
    clean = out / "step2_cleaned.csv"
    labelled = out / "step3_labelled.csv"
    train = out / "train.csv"
    test = out / "test.csv"

    print("\n" + "=" * 72 + f"\nDEVICE: {name}\n" + "=" * 72)

    # STEP 1 — Load extracted CSVs and attach labels
    print("\n[STEP 1] Load & inspect")
    sources = get_sources(folder)
    attack_count = len(sources) - 1
    print(f"  Attack CSV files found: {attack_count}")
    if attack_count == 0:
        print("  SKIPPED: no extracted attack CSVs available for this device.")
        return {"device": name, "status": "skipped_no_attack_csvs"}

    for p in [raw, clean, labelled, train, test]:
        if p.exists():
            p.unlink()

    first, raw_rows, source_report = True, 0, []
    for label, csv_path in sources:
        n = 0
        print(f"  Loading {label}: {csv_path.name}")
        for part in pd.read_csv(csv_path, chunksize=CHUNK_SIZE, low_memory=False):
            part["label"] = label
            write_frame(part, raw, first)
            first = False
            n += len(part)
            raw_rows += len(part)
        source_report.append({"label": label, "file": str(csv_path), "rows": n})
        print(f"    {n:,} rows")

    # STEP 2 — Convert features to numeric; replace inf; drop invalid rows
    print("\n[STEP 2] Clean")
    first, rows_read, rows_kept = True, 0, 0
    rows_dropped = pos_inf = neg_inf = 0
    feature_columns = None
    class_counts_clean = {}

    for part in pd.read_csv(raw, chunksize=CHUNK_SIZE, low_memory=False):
        features = [c for c in part.columns if c != "label"]
        if feature_columns is None:
            feature_columns = features
            print("  Feature columns:", len(features))

        numeric = part[features].apply(pd.to_numeric, errors="coerce")
        arr = numeric.to_numpy()
        pos_inf += int(np.isposinf(arr).sum())
        neg_inf += int(np.isneginf(arr).sum())
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
        labels = part["label"].astype("string")
        valid = numeric.notna().all(axis=1) & labels.notna() & labels.ne("")

        result = numeric.loc[valid].copy()
        result["label"] = labels.loc[valid].astype(str)
        rows_read += len(part)
        rows_kept += len(result)
        rows_dropped += len(part) - len(result)

        for key, value in result["label"].value_counts().items():
            class_counts_clean[key] = class_counts_clean.get(key, 0) + int(value)

        if len(result):
            write_frame(result, clean, first)
            first = False

    if rows_kept == 0:
        raise RuntimeError(f"{name}: no rows remain after cleaning.")
    print(f"  Read {rows_read:,} | kept {rows_kept:,} | dropped {rows_dropped:,}")
    print(f"  +Inf values: {pos_inf:,} | -Inf values: {neg_inf:,}")

    # Duplicate inspection within chunks only; no deletion.
    duplicate_count = 0
    for part in pd.read_csv(clean, chunksize=CHUNK_SIZE, low_memory=False):
        cols = [c for c in part.columns if c != "label"]
        duplicate_count += int(part.duplicated(subset=cols).sum())

    # STEP 3 — Add stable global label IDs
    print("\n[STEP 3] Prepare labels")
    first, labelled_rows, label_counts = True, 0, {}
    for part in pd.read_csv(clean, chunksize=CHUNK_SIZE, low_memory=False):
        part["label"] = part["label"].astype(str)
        unknown = sorted(set(part["label"].unique()) - set(LABEL_TO_ID))
        if unknown:
            raise ValueError(f"{name}: labels missing from global mapping: {unknown}")
        part["label_id"] = part["label"].map(LABEL_TO_ID).astype("int64")
        for key, value in part["label"].value_counts().items():
            label_counts[key] = label_counts.get(key, 0) + int(value)
        write_frame(part, labelled, first)
        first = False
        labelled_rows += len(part)
    print(f"  Encoded {labelled_rows:,} rows across {len(label_counts)} classes.")

    # STEP 4 — Exact stratified 80:20 split, using a seeded selection per class
    print("\n[STEP 4] Stratified train/test split")
    counts = {}
    for part in pd.read_csv(labelled, usecols=["label_id"], chunksize=CHUNK_SIZE):
        for key, value in part["label_id"].value_counts().items():
            key = int(key)
            counts[key] = counts.get(key, 0) + int(value)

    rng = np.random.default_rng(RANDOM_SEED)
    flags, expected_train, expected_test = {}, {}, {}
    for key, count in counts.items():
        if count < 2:
            raise ValueError(f"{name}: {ID_TO_LABEL[key]} has fewer than 2 rows.")
        n_train = min(max(int(round(count * TRAIN_RATIO)), 1), count - 1)
        selection = np.zeros(count, dtype=bool)
        selection[rng.choice(count, size=n_train, replace=False)] = True
        flags[key] = selection
        expected_train[key] = n_train
        expected_test[key] = count - n_train

    positions = {key: 0 for key in counts}
    first_train = first_test = True
    train_rows = test_rows = 0

    for part in pd.read_csv(labelled, chunksize=CHUNK_SIZE, low_memory=False):
        ids = part["label_id"].to_numpy()
        mask = np.zeros(len(part), dtype=bool)
        for key in np.unique(ids):
            key = int(key)
            idx = np.flatnonzero(ids == key)
            start, end = positions[key], positions[key] + len(idx)
            mask[idx] = flags[key][start:end]
            positions[key] = end

        train_part, test_part = part.loc[mask], part.loc[~mask]
        if len(train_part):
            write_frame(train_part, train, first_train)
            first_train = False
            train_rows += len(train_part)
        if len(test_part):
            write_frame(test_part, test, first_test)
            first_test = False
            test_rows += len(test_part)

    if any(positions[k] != counts[k] for k in counts):
        raise RuntimeError(f"{name}: split row count mismatch.")

    def count_ids(path):
        result, total = {}, 0
        for part in pd.read_csv(path, usecols=["label_id"], chunksize=CHUNK_SIZE):
            total += len(part)
            for key, value in part["label_id"].value_counts().items():
                key = int(key)
                result[key] = result.get(key, 0) + int(value)
        return total, result

    train_total, train_counts = count_ids(train)
    test_total, test_counts = count_ids(test)
    assert train_total + test_total == labelled_rows
    for key in counts:
        assert train_counts.get(key, 0) == expected_train[key]
        assert test_counts.get(key, 0) == expected_test[key]

    report = {
        "device": name,
        "status": "completed",
        "source_files": source_report,
        "feature_count": len(feature_columns),
        "step1_rows": raw_rows,
        "step2_rows_read": rows_read,
        "step2_rows_kept": rows_kept,
        "step2_rows_dropped": rows_dropped,
        "positive_infinity_values": pos_inf,
        "negative_infinity_values": neg_inf,
        "duplicate_feature_rows_within_chunks": duplicate_count,
        "step3_rows": labelled_rows,
        "train_rows": train_total,
        "test_rows": test_total,
        "train_ratio": TRAIN_RATIO,
        "random_seed": RANDOM_SEED,
        "class_distribution": {
            ID_TO_LABEL[key]: {
                "label_id": key,
                "total": counts[key],
                "train": train_counts.get(key, 0),
                "test": test_counts.get(key, 0)
            } for key in sorted(counts)
        },
        "note": "Within-device split, not cross-device evaluation. Duplicates were not removed. Fit scalers on training data only."
    }
    with open(out / "preprocessing_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print(f"  ✓ COMPLETE: {name} | train={train_total:,} | test={test_total:,}")
    return {
        "device": name, "status": "completed", "rows": labelled_rows,
        "train_rows": train_total, "test_rows": test_total,
        "classes": len(counts), "features": len(feature_columns)
    }


## 4. Run all discovered devices

This loop completes all four stages for one device before starting the next. A device with no extracted attack CSVs is recorded as skipped instead of silently being treated as complete.


In [ ]:
results = []
for device in devices:
    results.append(process_device(device))

summary = pd.DataFrame(results)
display(summary)

summary_path = OUTPUT_ROOT / "all_devices_preprocessing_summary.csv"
summary.to_csv(summary_path, index=False)

print("\nSummary saved to:", summary_path)
print("\nCheck any skipped devices and review each device's preprocessing_report.json.")


## 5. Output structure and final checks

For every successfully processed device, the notebook creates:

```text
preprocessed/
  Device_Name/
    step1_raw_labelled.csv
    step2_cleaned.csv
    step3_labelled.csv
    train.csv
    test.csv
    preprocessing_report.json
```

The root `preprocessed/` folder also contains `all_devices_preprocessing_summary.csv`.

Before moving to feature selection/model training:
- Confirm that all nine device folders were discovered.
- Investigate any device marked `skipped_no_attack_csvs`.
- Check feature counts (expected: 115) in each report.
- Check whether cleaning removed an unusually large fraction of rows.
- Note that some devices may legitimately lack Mirai or Gafgyt classes in the source dataset.

**Research note:** A stratified split inside each device measures within-device performance. A separate cross-device experiment is needed to test generalization to unseen IoT devices.
